# exp145_learned_likelihood_rawtest_feature_generator_parity train

Target-free full-train learned likelihood feature generator and schema parity audit.

## Contents

1. Setup and configuration
2. Input artifact checks
3. Full-train feature generation
4. Schema parity readout

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd

from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config
from learned_likelihood_rawtest_feature_generator_parity import run_generator

DEBUG = os.environ.get('EXPERIMENT_DEBUG', '0') == '1'
MAX_ROWS_ENV = os.environ.get('EXPERIMENT_MAX_ROWS')
MAX_ROWS = int(MAX_ROWS_ENV) if MAX_ROWS_ENV else (5000 if DEBUG else None)

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print('Experiment:', EXPERIMENT_NAME)
print('Route:', get_nested(config, 'experiment.route'))
print('Parent:', get_nested(config, 'lineage.parent'))
print('Output root:', paths.output_root)
print('Artifacts:', paths.artifacts_dir)
print('Debug:', DEBUG, 'Max rows:', MAX_ROWS)


## 2. Input artifact checks

In [ ]:
input_keys = [
    'data.exp099_train_feature_cache_local',
    'data.exp111_feature_schema',
    'data.exp111_model_manifest',
    'data.exp112_feature_schema',
]
for key in input_keys:
    value = get_nested(config, key)
    path = Path(value) if value else None
    print(key, '=>', path, 'exists=', bool(path and path.exists()))

print('Candidates:', [item['name'] for item in get_nested(config, 'generator.candidates')])
print('Expected output columns:', get_nested(config, 'audit.expected_feature_columns'))


## 3. Full-train feature generation

In [ ]:
summary = run_generator(
    output_dir=paths.artifacts_dir,
    mode='train',
    train_cache_path=get_nested(config, 'data.exp099_train_feature_cache_local'),
    rawtest_cache_path=None,
    exp111_schema_path=get_nested(config, 'data.exp111_feature_schema'),
    exp111_manifest_path=get_nested(config, 'data.exp111_model_manifest'),
    exp112_schema_path=get_nested(config, 'data.exp112_feature_schema'),
    max_rows=MAX_ROWS,
)

print(json.dumps(summary['generated_schema'], indent=2))


## 4. Schema parity readout

In [ ]:
parity_path = Path(summary['generated_schema']['parity_path'])
parity = pd.read_csv(parity_path)
print('schema parity pass:', summary['generated_schema']['schema_parity_pass'])
print('mismatch rows:', summary['generated_schema']['mismatch_rows'])
display(parity.head(20))

if not summary['generated_schema']['schema_parity_pass']:
    display(parity[~parity['matches_position']].head(50))
